In [2]:
import pandas as pd
import numpy as np
from collections import Counter
from scipy import sparse

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score

from data_preprocessing import create_train_test_val_sets, read_processed_data


# --------------------------------------------------
# Load processed datasets
# --------------------------------------------------
x_mendeley, y_mendeley = read_processed_data(r"..\data\processed\mendeley_processed.csv")
x_phiusiil, y_phiusiil = read_processed_data(r"..\data\processed\phiusiil_processed.csv")

# Fill missing values once here instead of using SimpleImputer in the pipeline.
# This avoids the large memory spikes you were seeing with sparse/high-dimensional data.
x_mendeley = x_mendeley.fillna(0)
x_phiusiil = x_phiusiil.fillna(0)

# Quick leakage sanity checks
print("Label in Mendeley features?", "Label" in x_mendeley.columns)
print("Label in Phiusiil features?", "Label" in x_phiusiil.columns)

# Create train/validation/test splits
# Use 3 folds instead of 5 to keep runtime manageable on a laptop.
mendeley_sets = create_train_test_val_sets(
    x_mendeley, y_mendeley, label_col="Label", test_size=0.2, n_splits=3
)
phiusiil_sets = create_train_test_val_sets(
    x_phiusiil, y_phiusiil, label_col="Label", test_size=0.2, n_splits=3
)


# --------------------------------------------------
# Utility: convert pandas DataFrame to CSR matrix
# --------------------------------------------------
def to_csr(X: pd.DataFrame) -> sparse.csr_matrix:
    """
    Convert a pandas DataFrame to a CSR sparse matrix.
    Works for both dense numeric frames and pandas sparse frames.
    """
    try:
        return X.sparse.to_coo().tocsr()
    except Exception:
        return sparse.csr_matrix(X.to_numpy(dtype=np.float32))


# Convert split datasets to sparse/numpy once to avoid repeated conversions later.
def prepare_dataset_for_modeling(dataset: dict) -> dict:
    return {
        "x_train_val": to_csr(dataset["x_train_val"]),
        "y_train_val": dataset["y_train_val"].reset_index(drop=True).to_numpy(),
        "x_test": to_csr(dataset["x_test"]),
        "y_test": dataset["y_test"].reset_index(drop=True).to_numpy(),
        "cv_splits": dataset["cv_splits"]
    }


mendeley_data = prepare_dataset_for_modeling(mendeley_sets)
phiusiil_data = prepare_dataset_for_modeling(phiusiil_sets)


# --------------------------------------------------
# Build sparse-friendly Logistic Regression pipeline
# --------------------------------------------------
def build_logistic_pipeline():
    """
    MaxAbsScaler is sparse-safe.
    saga is more suitable than lbfgs for large/sparse feature spaces.
    """
    return Pipeline([
        ("scaler", MaxAbsScaler()),
        ("model", LogisticRegression(
            solver="saga",
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        ))
    ])


# --------------------------------------------------
# Stratified subsample for fast tuning on large datasets
# --------------------------------------------------
def stratified_subsample(X, y, max_samples=None, random_state=42):
    """
    Return a stratified subsample for tuning.
    If max_samples is None or dataset is already small, return full data.
    """
    if max_samples is None or len(y) <= max_samples:
        return X, y

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        train_size=max_samples,
        random_state=random_state
    )
    sample_idx, _ = next(splitter.split(np.zeros(len(y)), y))
    return X[sample_idx], y[sample_idx]


# --------------------------------------------------
# Hyperparameter tuning
# --------------------------------------------------
def optimize_logistic_regression(X, y, max_tuning_samples=None) -> GridSearchCV:
    """
    Tune Logistic Regression on a full set or stratified sample.
    """
    X_tune, y_tune = stratified_subsample(X, y, max_samples=max_tuning_samples)

    params = {
        "model__C": [0.1, 1, 10]
    }

    inner_cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

    grid_search = GridSearchCV(
        estimator=build_logistic_pipeline(),
        param_grid=params,
        cv=inner_cv,
        scoring="f1",
        n_jobs=1,          # keeps results more reproducible across runs
        error_score="raise"
    )

    grid_search.fit(X_tune, y_tune)

    print("Best hyperparameters:")
    print(grid_search.best_params_)

    return grid_search


# --------------------------------------------------
# Nested CV evaluation + final hold-out test
# --------------------------------------------------
def train_no_feature_selection(dataset, dataset_name, max_tuning_samples=None):
    """
    Nested CV-style evaluation:
    - Outer CV: evaluation
    - Inner CV: hyperparameter tuning inside each fold
    Then:
    - choose the most common best C across folds
    - fit on full train_val
    - evaluate on hold-out test set
    """
    fold_scores = []
    all_y_val = []
    all_y_pred = []
    best_cs = []

    for fold_num, (train_idx, val_idx) in enumerate(dataset["cv_splits"], start=1):
        x_train = dataset["x_train_val"][train_idx]
        x_val = dataset["x_train_val"][val_idx]
        y_train = dataset["y_train_val"][train_idx]
        y_val = dataset["y_train_val"][val_idx]

        print(f"\n{dataset_name} Fold {fold_num}: tuning hyperparameters...")
        tuned_model = optimize_logistic_regression(
            x_train,
            y_train,
            max_tuning_samples=max_tuning_samples
        )

        best_c = tuned_model.best_params_["model__C"]
        best_cs.append(best_c)
        print(f"{dataset_name} Fold {fold_num} best C: {best_c}")

        y_pred = tuned_model.best_estimator_.predict(x_val)

        fold_scores.append(f1_score(y_val, y_pred))
        all_y_val.extend(y_val)
        all_y_pred.extend(y_pred)

    print(f"\n{dataset_name} Nested CV Results")
    print(f"Mean F1: {np.mean(fold_scores):.4f}, Std F1: {np.std(fold_scores):.4f}")
    print("Validation Results:")
    print(classification_report(all_y_val, all_y_pred))

    # Use the most common best C from the outer folds for final test evaluation
    final_c = Counter(best_cs).most_common(1)[0][0]
    print(f"{dataset_name} Final chosen C for hold-out test: {final_c}")

    final_model = Pipeline([
        ("scaler", MaxAbsScaler()),
        ("model", LogisticRegression(
            C=final_c,
            solver="saga",
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        ))
    ])

    final_model.fit(dataset["x_train_val"], dataset["y_train_val"])
    y_test_pred = final_model.predict(dataset["x_test"])

    print(f"\n{dataset_name} Test Set Results:")
    print(classification_report(dataset["y_test"], y_test_pred))


# --------------------------------------------------
# Run experiments
# --------------------------------------------------
print("\n========== Mendeley ==========")
train_no_feature_selection(
    mendeley_data,
    "Mendeley",
    max_tuning_samples=None   # full train fold is fine here
)

print("\n========== Phiusiil ==========")
train_no_feature_selection(
    phiusiil_data,
    "Phiusiil",
    max_tuning_samples=50000  # keeps tuning fast on your laptop
)

Label in Mendeley features? False
Label in Phiusiil features? False
Train/validation/test split prepared: 198360 instances for training and validation, 49590 instances for testing
Stratified 3-fold CV splits created.
Train/validation/test split prepared: 188636 instances for training and validation, 47159 instances for testing
Stratified 3-fold CV splits created.

========== Mendeley ==========

Mendeley Fold 1: tuning hyperparameters...
Best hyperparameters:
{'model__C': 10}
Mendeley Fold 1 best C: 10

Mendeley Fold 2: tuning hyperparameters...
Best hyperparameters:
{'model__C': 10}
Mendeley Fold 2 best C: 10

Mendeley Fold 3: tuning hyperparameters...
Best hyperparameters:
{'model__C': 10}
Mendeley Fold 3 best C: 10

Mendeley Nested CV Results
Mean F1: 0.7856, Std F1: 0.0002
Validation Results:
              precision    recall  f1-score   support

           0       0.78      0.86      0.82    102833
           1       0.83      0.74      0.79     95527

    accuracy                

C:\Users\ovaze\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
